# Sphere Model — yamc CPU vs yamc GPU

Tracks how close the GPU dispatch's flux spectrum is to the full-physics CPU result, on the same model. The GPU kernel currently implements **elastic scatter + capture only**; the CPU runs full physics (inelastic, fission, URR, secondary distributions, …). This notebook is **not** an equivalence demo today (CPU and GPU don't agree yet) — it's a regression-style measure that should improve as kernel physics is added.

## Why a separate notebook from `tasks/neutron/sphere/endf-b8.1.ipynb`?

The yamc-vs-OpenMC sphere notebook uses three features the GPU dispatch rejects today:

- VITAMIN-J-175 energy group structure (not log-uniform → `EnergyBinsNotLogUniform`).
- One tally per MT plus heating/production (→ `MultipleTalliesNotSupported`).
- `particle="neutron"` filter (→ `UnsupportedTallyFilters`).

So this notebook lives separately and uses a GPU-compatible subset (single Flux tally, log-uniform bins, no particle filter). When the dispatch widens to handle those features, the comparison can be folded back into the main notebook as a third leg (openmc | yamc-cpu | yamc-gpu) and this file deleted.

## Model

- **Geometry**: 35 cm radius sphere, vacuum boundary.
- **Source**: Point source at origin, 14.06 MeV monoenergetic neutrons, isotropic.
- **Material**: Single nuclide at 1.0 g/cm³, 294 K. Same shape as the existing yamc-vs-OpenMC notebook so results are comparable.
- **Tally**: One Flux tally with `[CellFilter, EnergyFilter]` over log-uniform energy bins.

## Output

Per nuclide:
- Flux spectrum on log-log axes with MC error bars.
- GPU/CPU ratio per bin with combined error bars.
- Bin-summed integral ratio.
- χ² per d.o.f. of bin-by-bin residuals weighted by combined std-dev — the headline regression number.

## Configuration via env vars

| Env var | Values | Default | Effect |
|---|---|---|---|
| `YAMC_VERIFY_COMPUTE_MODE` | `cpu`, `gpu`, `both` | `both` | Which backends to run. `both` produces the comparison plots; `cpu`/`gpu` only run that backend (useful for benchmarking or when the other backend isn't available). |
| `YAMC_VERIFY_NUCLIDES` | comma list, e.g. `H1,Be9,Fe56` | `H1,Be9,Fe56,W184,Pb208` | Override the nuclide list. |
| `YAMC_VERIFY_NUMBER_OF_SIMULATIONS` | int | (no cap) | Cap whichever list is in use to the first N entries (smoke-test pattern, matches the sibling notebooks). |

## Skip behaviour

When `compute_mode` includes `gpu`, the notebook errors with a clear message if no f64 GPU is available — `yamc.gpu_available()` is the gate. CI runners without a GPU should run with `YAMC_VERIFY_COMPUTE_MODE=cpu` to skip the GPU path.

In [ ]:
# Setup and imports
import math
import os
import time

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

import yamc

# Configuration
PARTICLES = 50_000
BATCHES = 4
SOURCE_ENERGY = 14.06e6  # eV
SPHERE_RADIUS_CM = 35.0
DENSITY_G_PER_CM3 = 1.0
TEMPERATURE_K = 294
N_ENERGY_BINS = 50
ENERGY_MIN_EV = 1.0e3
ENERGY_MAX_EV = 2.0e7  # > source energy so the source bin isn't on the edge
MAX_PLOT_ENERGY = SOURCE_ENERGY * 1.1
FLUX_DECADES = 6  # y-axis range below max value, in decades

DEFAULT_NUCLIDES = ["H1", "Be9", "Fe56", "W184", "Pb208"]

_VALID_MODES = {"cpu", "gpu", "both"}
COMPUTE_MODE = os.environ.get("YAMC_VERIFY_COMPUTE_MODE", "both").strip().lower()
if COMPUTE_MODE not in _VALID_MODES:
    raise ValueError(
        f"YAMC_VERIFY_COMPUTE_MODE must be one of {sorted(_VALID_MODES)}; "
        f"got {COMPUTE_MODE!r}"
    )
RUN_CPU = COMPUTE_MODE in ("cpu", "both")
RUN_GPU = COMPUTE_MODE in ("gpu", "both")

_env_nuclides = os.environ.get("YAMC_VERIFY_NUCLIDES")
NUCLIDES = (
    [n.strip() for n in _env_nuclides.split(",") if n.strip()]
    if _env_nuclides
    else list(DEFAULT_NUCLIDES)
)
_cap = int(os.environ.get("YAMC_VERIFY_NUMBER_OF_SIMULATIONS", "0")) or None
if _cap:
    NUCLIDES = NUCLIDES[:_cap]

if RUN_GPU and not yamc.gpu_available():
    raise RuntimeError(
        "YAMC_VERIFY_COMPUTE_MODE includes 'gpu' but `yamc.gpu_available()` "
        "returned False — either the host has no f64 Vulkan adapter, or yamc "
        "was built without `--features ...,gpu`. Set YAMC_VERIFY_COMPUTE_MODE=cpu "
        "to skip the GPU run."
    )

print(f"Compute:     {COMPUTE_MODE} (CPU={RUN_CPU}, GPU={RUN_GPU})")
print(f"Nuclides:    {NUCLIDES}")
print(f"Particles:   {PARTICLES} × {BATCHES} batches = {PARTICLES * BATCHES} histories")
print(f"Energy bins: {N_ENERGY_BINS} log-uniform from {ENERGY_MIN_EV:g} to {ENERGY_MAX_EV:g} eV")

In [ ]:
def _log_uniform_bins(e_min, e_max, n_bins):
    """N+1 boundaries uniform in ln(E). Matches what the GPU kernel
    produces internally; the dispatch validates this exact spacing."""
    log_min, log_max = math.log(e_min), math.log(e_max)
    return [
        math.exp(log_min + i * (log_max - log_min) / n_bins)
        for i in range(n_bins + 1)
    ]


def run_yamc_simulation(nuclide, compute):
    """Run a yamc simulation for one nuclide on one backend.

    Returns a dict with `mean`, `std_dev`, `energy_bins`, `particles_per_second`,
    `elapsed_secs`. Same model on both backends — only `compute=` differs.
    """
    sphere = yamc.Sphere(
        x0=0.0, y0=0.0, z0=0.0,
        radius=SPHERE_RADIUS_CM,
        boundary="vacuum",
    )
    region = -sphere

    material = yamc.Material(
        composition={nuclide: 1.0},
        density=DENSITY_G_PER_CM3,
        temperature=TEMPERATURE_K,
    )
    material.read_nuclear_data("endf-b8.1")

    cell = yamc.Cell(region=region, material=material, name="sphere")
    geometry = yamc.Geometry([cell])

    bins = _log_uniform_bins(ENERGY_MIN_EV, ENERGY_MAX_EV, N_ENERGY_BINS)
    tally = yamc.Tally(
        scores=["flux"],
        cells=cell,
        energy_bins=bins,
        name="flux_spectrum",
    )

    source = yamc.NeutronSource(
        position=yamc.Point(0, 0, 0),
        energy=yamc.Discrete([SOURCE_ENERGY], [1.0]),
    )

    model = yamc.Model(
        geometry=geometry,
        tallies=[tally],
        source=source,
        particles_per_batch=PARTICLES,
        batches=BATCHES,
        seed=1,
    )
    t0 = time.perf_counter()
    results = model.simulate_transport(compute=compute)
    elapsed = time.perf_counter() - t0

    r = results[tally]
    return {
        "mean": np.array(r.mean).squeeze(),
        "std_dev": np.array(r.standard_deviation).squeeze(),
        "energy_bins": np.array(bins),
        "particles_per_second": (PARTICLES * BATCHES) / elapsed,
        "elapsed_secs": elapsed,
    }

In [ ]:
def _step_trace(energy_bins, values, name, color, dash=None):
    """Step (staircase) trace for Plotly — same trick as the existing
    sphere/endf-b8.1 notebook so plots look familiar."""
    x = []
    y = []
    for i in range(len(values)):
        x.extend([energy_bins[i], energy_bins[i + 1]])
        y.extend([values[i], values[i]])
    return go.Scatter(
        x=x, y=y, mode="lines", name=name,
        line=dict(color=color, width=2, dash=dash),
    )


def _chi2_per_dof(cpu, gpu):
    """χ² / d.o.f. of bin-by-bin residuals, weighted by combined std-dev.
    Bins where both std-devs are zero are skipped (no information). The
    headline regression number — drops as the kernel's physics improves.
    """
    diff = gpu["mean"] - cpu["mean"]
    var = cpu["std_dev"] ** 2 + gpu["std_dev"] ** 2
    mask = var > 0
    if not np.any(mask):
        return float("nan"), 0
    chi2 = float(np.sum(diff[mask] ** 2 / var[mask]))
    dof = int(np.sum(mask))
    return chi2 / dof, dof


def make_comparison_plot(nuclide, cpu, gpu):
    """Two-panel Plotly figure: flux spectrum (top), GPU/CPU ratio (bottom)."""
    bins = cpu["energy_bins"]
    bin_centers = np.sqrt(bins[:-1] * bins[1:])  # geometric mean for log scale

    # Normalise by lethargy bin width so the spectrum isn't dominated by
    # the bin-width gradient on log-uniform bins.
    lethargy = np.log(bins[1:] / bins[:-1])
    cpu_flux = cpu["mean"] / lethargy
    cpu_err = cpu["std_dev"] / lethargy
    gpu_flux = gpu["mean"] / lethargy
    gpu_err = gpu["std_dev"] / lethargy

    # GPU / CPU ratio with combined relative error
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(cpu_flux > 0, gpu_flux / cpu_flux, np.nan)
        rel_err = np.where(
            (cpu_flux > 0) & (gpu_flux > 0),
            np.sqrt(
                (cpu_err / np.where(cpu_flux > 0, cpu_flux, 1.0)) ** 2
                + (gpu_err / np.where(gpu_flux > 0, gpu_flux, 1.0)) ** 2
            ),
            np.nan,
        )
        ratio_err = np.where(np.isfinite(ratio), ratio * rel_err, np.nan)

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[0.7, 0.3],
    )
    fig.add_trace(_step_trace(bins, cpu_flux, "yamc CPU", "#1f77b4"), row=1, col=1)
    fig.add_trace(_step_trace(bins, gpu_flux, "yamc GPU", "#ff7f0e", dash="dash"), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=bin_centers, y=cpu_flux, error_y=dict(type="data", array=cpu_err, visible=True),
        mode="markers", marker=dict(color="#1f77b4", size=4),
        showlegend=False, name="CPU error",
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=bin_centers, y=gpu_flux, error_y=dict(type="data", array=gpu_err, visible=True),
        mode="markers", marker=dict(color="#ff7f0e", size=4),
        showlegend=False, name="GPU error",
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=bin_centers, y=ratio,
        error_y=dict(type="data", array=ratio_err, visible=True),
        mode="markers", marker=dict(color="#2ca02c", size=5),
        showlegend=False, name="GPU/CPU",
    ), row=2, col=1)
    fig.add_hline(y=1.0, line_dash="dot", line_color="gray", row=2, col=1)

    # Axis ranges + scales
    fig.update_xaxes(type="log", title="Energy (eV)", row=2, col=1)
    max_flux = float(np.nanmax(np.maximum(cpu_flux, gpu_flux)))
    if max_flux > 0:
        log_max = math.log10(max_flux)
        fig.update_yaxes(
            type="log", title="Flux per unit lethargy",
            range=[log_max - FLUX_DECADES, log_max + 0.1],
            row=1, col=1,
        )
    fig.update_yaxes(title="GPU / CPU", range=[0, 2.0], row=2, col=1)

    chi2, dof = _chi2_per_dof(cpu, gpu)
    cpu_total = float(np.sum(cpu["mean"]))
    gpu_total = float(np.sum(gpu["mean"]))
    integral_ratio = gpu_total / cpu_total if cpu_total > 0 else float("nan")
    fig.update_layout(
        title=(
            f"<b>{nuclide}</b> &nbsp; "
            f"χ²/dof = {chi2:.2f} ({dof} bins) &nbsp; "
            f"∫gpu/∫cpu = {integral_ratio:.3f} &nbsp; "
            f"CPU {cpu['particles_per_second']:.0f} p/s &nbsp; "
            f"GPU {gpu['particles_per_second']:.0f} p/s"
        ),
        height=600, hovermode="x unified",
    )
    return fig, chi2, dof, integral_ratio

In [ ]:
# Run all simulations and collect results.
all_results = {}
for nuclide in NUCLIDES:
    print(f"--- {nuclide} ---")
    res = {}
    if RUN_CPU:
        cpu = run_yamc_simulation(nuclide, compute="cpu")
        print(f"  CPU: {cpu['particles_per_second']:.0f} p/s")
        res["cpu"] = cpu
    if RUN_GPU:
        gpu = run_yamc_simulation(nuclide, compute="gpu")
        print(f"  GPU: {gpu['particles_per_second']:.0f} p/s")
        res["gpu"] = gpu
    all_results[nuclide] = res

## Per-nuclide results

For each nuclide: flux spectrum CPU vs GPU (top) and ratio (bottom). The χ²/dof in the title is the headline regression number — **lower is better**, and it should drop monotonically as kernel physics improves. The ∫gpu/∫cpu integral ratio is a coarser sanity check; bin-by-bin agreement is what χ²/dof captures.

In [ ]:
# Display per-nuclide results.
summary_rows = []
for nuclide, res in all_results.items():
    cpu = res.get("cpu")
    gpu = res.get("gpu")
    if cpu is not None and gpu is not None:
        # Both backends ran — show the comparison plot.
        fig, chi2, dof, integral_ratio = make_comparison_plot(nuclide, cpu, gpu)
        fig.show()
        summary_rows.append({
            "nuclide": nuclide,
            "chi2_per_dof": chi2,
            "dof": dof,
            "integral_ratio_gpu_cpu": integral_ratio,
            "cpu_particles_per_s": cpu["particles_per_second"],
            "gpu_particles_per_s": gpu["particles_per_second"],
            "gpu_speedup": gpu["particles_per_second"] / cpu["particles_per_second"],
        })
    else:
        # Single-backend run — no comparison; just print throughput.
        backend, data = ("cpu", cpu) if cpu is not None else ("gpu", gpu)
        display(Markdown(f"### {nuclide} ({backend} only)"))
        display(Markdown(f"- **{backend.upper()} throughput:** {data['particles_per_second']:.0f} p/s"))
        summary_rows.append({
            "nuclide": nuclide,
            "backend": backend,
            "particles_per_s": data["particles_per_second"],
            "elapsed_secs": data["elapsed_secs"],
        })

import pandas as pd
summary = pd.DataFrame(summary_rows)
display(Markdown("## Summary"))
if {"chi2_per_dof", "gpu_speedup"}.issubset(summary.columns):
    display(summary.style.format({
        "chi2_per_dof": "{:.2f}",
        "integral_ratio_gpu_cpu": "{:.3f}",
        "cpu_particles_per_s": "{:.0f}",
        "gpu_particles_per_s": "{:.0f}",
        "gpu_speedup": "{:.1f}×",
    }))
else:
    display(summary.style.format({
        "particles_per_s": "{:.0f}",
        "elapsed_secs": "{:.2f}s",
    }))